In [1]:
from pathlib import Path
import pandas as pd

# Check the available datasets and their columns
dataset_names = [
    "territorial_associations.csv",
    "directory_organizations.csv",
    "healthcare_services.csv",
    "department_violence.csv",
    "minor_intrafamily_violence.csv",
    "state_funding_2020_2024.csv"
]

for name in dataset_names:
    file_path = Path("data/processed") / name
    columns = pd.read_csv(file_path, nrows=0).columns.tolist()

    print(f"\n{name}")
    print(columns)


territorial_associations.csv
['id', 'siret', 'titre', 'objet', 'date_creat', 'creation_year', 'adrs_codeinsee', 'adrs_libcommune', 'postal_code', 'department', 'siteweb', 'position', 'VSS_VIF', 'WOMEN_EQUALITY']

directory_organizations.csv
['name_clean', 'name', 'departments_covered', 'data_source', 'rna_id', 'head_office_verified', 'exact_name_in_rna']

healthcare_services.csv
['name', 'sourceRef', 'email', 'phone', 'phoneSecondary', 'website', 'address', 'addressComplement', 'postalCode', 'city', 'latitude', 'longitude', 'key', 'violenceTypes', 'id_directory', 'sourceName', 'department', 'name_clean', 'id_rna', 'titre', 'suggested_match']

department_violence.csv
['department_code', 'region_code', 'year', 'violence_type', 'count_unit', 'recorded_victims', 'rate_per_1000', 'population', 'population_reference_year', 'housing_units', 'housing_reference_year', 'department']

minor_intrafamily_violence.csv
['department_code', 'department', 'rate_per_1000_minors', 'violence_type', 'perio

In [2]:
# Read identifiers as text to preserve leading zeros

associations = pd.read_csv("data/processed/territorial_associations.csv",
    dtype="string")

funding = pd.read_csv("data/processed/state_funding_2020_2024.csv",
    dtype="string")

# Check association identifiers
print("Missing RNA IDs:", associations["id"].isna().sum())
print("Duplicate RNA IDs:", associations["id"].duplicated().sum())
print("Missing association SIRETs:", associations["siret"].isna().sum())

# Count associations with a SIRET found in the funding data
matched = (associations["siret"].notna()
           & associations["siret"].isin(funding["siret"].dropna()))

print("Associations with a funding SIRET match:", matched.sum())

Missing RNA IDs: 0
Duplicate RNA IDs: 0
Missing association SIRETs: 499
Associations with a funding SIRET match: 0


In [3]:
# Inspect the available association SIRET

display(associations.loc[associations["siret"].notna(),
    ["titre", "siret", "department"]])

,titre,siret,department
3,INPULSE-ADELPHIE,84781921600015.0,38
10,S.O.S. INCESTE POUR REVIVRE,38985346600024.0,38
33,MILLE MONTS,80061274900013.0,38
37,L'ECHAPPEE,81360948400015.0,38
40,SOLENCIEL,82925647800027.0,38
46,SEREIN.E.S,87793023000025.0,38
80,ONG COEUR D'HUMANITÉ INTERNATIONAL,94349098700014.0,38
114,FORUM GAY ET LESBIEN DE LYON,41016145900020.0,69
120,COMITE DEPARTEMENTAL DE L'UNION FRANCAISE DES ...,43294958400010.0,69
135,HFX+ AUVERGNE-RHÔNE-ALPES,52175388900026.0,69


In [4]:
# Remove the numeric suffix from association SIRET
associations["siret"] = associations["siret"].str.replace(r"\.0$", "", regex=True)

# Check the format and compare again
valid_siret = associations["siret"].str.fullmatch(r"[0-9]{14}", na=False)

matched = (valid_siret& associations["siret"].isin(funding["siret"].dropna()))

print("Valid 14-digit SIRETs:", valid_siret.sum())
print("Associations with a funding SIRET match:", matched.sum())

Valid 14-digit SIRETs: 23
Associations with a funding SIRET match: 9


## Database relationships

- RNA ID are complete and unique in the association dataset.
- Only 23 of the 522 associations have a SIRET.
- After correcting the SIRET format, 9 associations match at least
  one record in the state funding dataset.
- A missing match does not mean that an association receives no funding.
- Department codes allow territorial comparisons between datasets.
  These comparisons do not establish a direct link between funding
  and recorded violence.
- Directory organizations and healthcare services remain separate
  because their identifiers and verification status differ.

In [5]:
# Match CSV column names to the SQL table
associations_sql = associations.rename(columns={
    "id": "rna_id","titre": "association_name","objet": "association_purpose","date_creat": "creation_date","adrs_codeinsee": "municipality_code",
    "adrs_libcommune": "municipality_name","department": "department_code","siteweb": "website","position": "administrative_status",
    "VSS_VIF": "vss_vif","WOMEN_EQUALITY": "women_equality"}).copy()

print(associations_sql.shape)
print(associations_sql.columns.tolist())

(522, 14)
['rna_id', 'siret', 'association_name', 'association_purpose', 'creation_date', 'creation_year', 'municipality_code', 'municipality_name', 'postal_code', 'department_code', 'website', 'administrative_status', 'vss_vif', 'women_equality']


In [6]:
# Check values that need a specific SQL format
print("Creation dates:")
print(associations_sql["creation_date"].head())

print("\nVSS/VIF values:")
print(associations_sql["vss_vif"].unique())

print("\nWomen / equality values:")
print(associations_sql["women_equality"].unique())

Creation dates:
0    1981-06-23
1    2007-04-03
2    1992-08-07
3    2007-06-11
4    2007-06-19
Name: creation_date, dtype: string

VSS/VIF values:
<StringArray>
['True', 'False']
Length: 2, dtype: string

Women / equality values:
<StringArray>
['False', 'True']
Length: 2, dtype: string


In [7]:
# Validate dates and keep the SQL date format
associations_sql["creation_date"] = pd.to_datetime(associations_sql["creation_date"],format="%Y-%m-%d",errors="raise").dt.strftime("%Y-%m-%d")

# Convert text indicators to 1 and 0
for column in ["vss_vif", "women_equality"]:
    associations_sql[column] = associations_sql[column].map({
        "True": 1,"False": 0})
display(associations_sql[["creation_date", "vss_vif", "women_equality"]].head())

,creation_date,vss_vif,women_equality
0,1981-06-23,1,0
1,2007-04-03,1,0
2,1992-08-07,0,1
3,2007-06-11,0,1
4,2007-06-19,0,1


In [9]:
from getpass import getpass
from sqlalchemy import create_engine, URL, text

# Ask for the password without writing it in the notebook
mysql_password = getpass("MySQL password: ")

# Connect to the local project database
connection_url = URL.create("mysql+pymysql",username="root",password=mysql_password,host="localhost",port=3306,database="cidff_project",
                            query={"charset": "utf8mb4"})

engine = create_engine(connection_url)

# Check the connection
with engine.connect() as connection:
    print("Connected to:", connection.execute(text("SELECT DATABASE()")).scalar())

MySQL password:  ········


Connected to: cidff_project


In [10]:
# Remove numeric suffixes from postal codes
associations_sql["postal_code"] = (associations_sql["postal_code"].str.strip().str.replace(r"\.0$", "", regex=True))

# Display any remaining invalid postal codes
invalid_postal = (associations_sql["postal_code"].notna()& ~associations_sql["postal_code"].str.fullmatch(r"[0-9]{5}", na=False))

display(associations_sql.loc[invalid_postal,["association_name", "postal_code"]])

,association_name,postal_code


In [11]:
# Import associations only if the SQL table is empty
with engine.begin() as connection:
    row_count = connection.execute(
        text("SELECT COUNT(*) FROM associations")
    ).scalar()

    if row_count == 0:
        associations_sql.to_sql(
            "associations",
            con=connection,
            if_exists="append",
            index=False
        )
        print("Imported:", len(associations_sql), "associations")
    else:
        print("Import skipped: the table already contains", row_count, "rows")

Import skipped: the table already contains 522 rows


In [12]:
# Read department codes as text
department_violence = pd.read_csv("data/processed/department_violence.csv",dtype={"department_code": "string"})

# Check the proposed primary key
key_columns = ["department_code", "year", "violence_type"]

print("Rows:", len(department_violence))
print("Rows with a missing key value:",department_violence[key_columns].isna().any(axis=1).sum())
print("Duplicate keys:",department_violence.duplicated(subset=key_columns).sum())

print("\nColumn types:")
print(department_violence.dtypes)

Rows: 80
Rows with a missing key value: 0
Duplicate keys: 0

Column types:
department_code               string
region_code                    int64
year                           int64
violence_type                    str
count_unit                       str
recorded_victims               int64
rate_per_1000                float64
population                     int64
population_reference_year      int64
housing_units                  int64
housing_reference_year         int64
department                       str
dtype: object


In [13]:
# Keep the department name in the reference table only
department_violence_sql = department_violence.drop(columns=["department"]).copy()

# Import only if the SQL table is empty
with engine.begin() as connection:
    row_count = connection.execute(
        text("SELECT COUNT(*) FROM department_violence")).scalar()

    if row_count == 0:
        department_violence_sql.to_sql("department_violence",con=connection,
            if_exists="append",index=False)
        print("Imported:", len(department_violence_sql), "records")
    else:
        print("Import skipped: the table already contains", row_count, "rows")

Import skipped: the table already contains 80 rows


In [14]:
# Load average rates of intrafamily violence against minors
minor_violence = pd.read_csv("data/processed/minor_intrafamily_violence.csv",dtype={"department_code": "string"})

# Check the proposed primary key
key_columns = ["department_code", "period", "violence_type"]

print("Rows:", len(minor_violence))
print("Rows with a missing key value:",minor_violence[key_columns].isna().any(axis=1).sum())
print("Duplicate keys:",minor_violence.duplicated(subset=key_columns).sum())

display(minor_violence.head())

Rows: 8
Rows with a missing key value: 0
Duplicate keys: 0


,department_code,department,rate_per_1000_minors,violence_type,period
0,38,Isère,3.463689,Physical violence,2022-2024
1,69,Rhône,2.147580,Physical violence,2022-2024
2,73,Savoie,2.882646,Physical violence,2022-2024
3,74,Haute-Savoie,2.289169,Physical violence,2022-2024
4,38,Isère,1.172379,Sexual violence,2022-2024


In [15]:
# Remove the department name before importing
minor_violence_sql = minor_violence.drop(columns=["department"]).copy()

# Check the table and import within the same transaction
with engine.begin() as connection:
    row_count = connection.execute(
        text("SELECT COUNT(*) FROM minor_intrafamily_violence")).scalar()

    if row_count == 0:
        minor_violence_sql.to_sql(
            "minor_intrafamily_violence", con=connection,
            if_exists="append", index=False)
        print("Imported:", len(minor_violence_sql), "records")
    else:
        print("Import skipped:", row_count, "rows already present")

Import skipped: 8 rows already present


In [16]:
# Load funding data with numeric amounts and text identifiers
funding_sql = pd.read_csv("data/processed/state_funding_2020_2024.csv",dtype={"department_code": "string","municipality_code": "string",
        "siren": "string","nic": "string","siret": "string","budget_program": "string","siren_original": "string"})

print("Rows:", len(funding_sql))
print("\nColumn types:")
print(funding_sql.dtypes)
print("\nMissing amounts:", funding_sql["amount"].isna().sum())

Rows: 30309

Column types:
year                            int64
department_code                string
municipality_code              string
municipality_name                 str
siren                          string
nic                            string
association_name                  str
budget_program                 string
amount                        float64
funding_purpose                   str
agreement                         str
siren_original                 string
siren_corrected                  bool
siren_format_valid               bool
geography_corrected              bool
identical_record_flag            bool
siret                          string
historical_purpose_context        str
dtype: object

Missing amounts: 0


In [17]:
# Import funding records only if the SQL table is empty
with engine.begin() as connection:
    row_count = connection.execute(text("SELECT COUNT(*) FROM funding_payments")).scalar()

    if row_count == 0:
        funding_sql.to_sql(
            "funding_payments",
            con=connection,
            if_exists="append",
            index=False,
            chunksize=1000)
        print("Imported:", len(funding_sql), "funding records")
    else:
        print("Import skipped:", row_count, "rows already present")

Import skipped: 30309 rows already present


In [18]:
# Compare annual totals before and after import
csv_totals = funding_sql.groupby("year")["amount"].sum()

sql_totals = pd.read_sql(
    """
    SELECT year, SUM(amount) AS amount
    FROM funding_payments
    GROUP BY year
    """,
    engine
).set_index("year")["amount"]

comparison = pd.DataFrame({
    "csv_amount": csv_totals,
    "sql_amount": sql_totals
})

comparison["difference"] = (
    comparison["sql_amount"] - comparison["csv_amount"]
).round(2)

display(comparison)

,csv_amount,sql_amount,difference
year,,,
2020,4.733410e+08,4.733410e+08,0.0
2021,4.905178e+08,4.905178e+08,0.0
2022,5.150348e+08,5.150348e+08,0.0
2023,5.552485e+08,5.552485e+08,0.0
2024,6.004853e+08,6.004853e+08,0.0


In [19]:
# Load the additional territorial datasets
directory = pd.read_csv("data/processed/directory_organizations.csv",dtype="string")

healthcare = pd.read_csv("data/processed/healthcare_services.csv",dtype="string")

# Check identifiers and geographical information
display(directory[["name", "departments_covered", "rna_id", "data_source"]])

display(healthcare[["name", "id_directory", "department", "postalCode"]])

,name,departments_covered,rna_id,data_source
0,Addcaes,73,<NA>,Arretons les violences
1,Alpa,38,<NA>,Arretons les violences
2,Avij Des Savoie,"73, 74",<NA>,Arretons les violences
3,Isis,38,<NA>,Arretons les violences
4,Issue De Secours - Le Rialto,38,<NA>,Arretons les violences
5,La Sasson,73,<NA>,Arretons les violences
6,Pluri-Elles,38,<NA>,Arretons les violences
7,Rivhaj,38,<NA>,Arretons les violences
8,Tandem,69,<NA>,Arretons les violences
9,Viffilavi France Victimes 69,69,<NA>,Arretons les violences


,name,id_directory,department,postalCode
0,Maison Des Femmes / Santé De L'Isère,1785,38,38700
1,Maison Des Femmes / Santé De Savoie,1908,73,73000
2,Maison Des Femmes / Santé De Haute-Savoie,1909,74,74130
3,Maison Des Femmes / Santé Du Rhône,2009,69,69003


In [20]:
# Select and rename the directory columns
directory_sql = directory[["name", "data_source", "rna_id", "head_office_verified"]].rename(columns={"name": "organization_name"}).copy()

# Assign IDs that will also be used in the department links
directory_sql.insert(0, "directory_id", range(1, len(directory_sql) + 1))

# Convert the verification indicator to 1 or 0
directory_sql["head_office_verified"] = (directory_sql["head_office_verified"].map({"True": 1, "False": 0}).astype("Int64"))

display(directory_sql)

,directory_id,organization_name,data_source,rna_id,head_office_verified
0,1,Addcaes,Arretons les violences,<NA>,0
1,2,Alpa,Arretons les violences,<NA>,0
2,3,Avij Des Savoie,Arretons les violences,<NA>,0
3,4,Isis,Arretons les violences,<NA>,0
4,5,Issue De Secours - Le Rialto,Arretons les violences,<NA>,0
5,6,La Sasson,Arretons les violences,<NA>,0
6,7,Pluri-Elles,Arretons les violences,<NA>,0
7,8,Rivhaj,Arretons les violences,<NA>,0
8,9,Tandem,Arretons les violences,<NA>,0
9,10,Viffilavi France Victimes 69,Arretons les violences,<NA>,0


In [21]:
# Import directory organizations only if the table is empty
with engine.begin() as connection:
    row_count = connection.execute(text("SELECT COUNT(*) FROM directory_organizations")).scalar()

    if row_count == 0:
        directory_sql.to_sql("directory_organizations",con=connection,
            if_exists="append",
            index=False
        )
        print("Imported:", len(directory_sql), "organizations")
    else:
        print("Import skipped:", row_count, "rows already present")

Imported: 12 organizations


In [23]:
# Keep each organization's ID and covered departments
directory_links = directory[["departments_covered"]].copy()
directory_links["directory_id"] = directory_sql["directory_id"]

# Create one row per covered department
directory_links["department_code"] = (
    directory_links["departments_covered"].str.split(",")
)

directory_links = directory_links.explode("department_code")
directory_links["department_code"] = (
    directory_links["department_code"].str.strip()
)

directory_links = directory_links[
    ["directory_id", "department_code"]
].reset_index(drop=True)

display(directory_links)
print("Department links:", len(directory_links))

,directory_id,department_code
0,1,73
1,2,38
2,3,73
3,3,74
4,4,38
5,5,38
6,6,73
7,7,38
8,8,38
9,9,69


Department links: 14


In [24]:
# Import department links only if the table is empty
with engine.begin() as connection:
    row_count = connection.execute(
        text("SELECT COUNT(*) FROM directory_departments")
    ).scalar()

    if row_count == 0:
        directory_links.to_sql(
            "directory_departments",
            con=connection,
            if_exists="append",
            index=False
        )
        print("Imported:", len(directory_links), "department links")
    else:
        print("Import skipped:", row_count, "rows already present")

Imported: 14 department links


In [25]:
# Select and rename healthcare service columns
healthcare_sql = healthcare[["id_directory", "name", "department", "postalCode"]].rename(columns={
    "id_directory": "directory_service_id",
    "name": "service_name",
    "department": "department_code",
    "postalCode": "postal_code"}).copy()

display(healthcare_sql)

,directory_service_id,service_name,department_code,postal_code
0,1785,Maison Des Femmes / Santé De L'Isère,38,38700
1,1908,Maison Des Femmes / Santé De Savoie,73,73000
2,1909,Maison Des Femmes / Santé De Haute-Savoie,74,74130
3,2009,Maison Des Femmes / Santé Du Rhône,69,69003


In [26]:
# Import healthcare services only if the table is empty
with engine.begin() as connection:
    row_count = connection.execute(text("SELECT COUNT(*) FROM healthcare_services")).scalar()

    if row_count == 0:
        healthcare_sql.to_sql(
            "healthcare_services",
            con=connection,
            if_exists="append",
            index=False
        )
        print("Imported:", len(healthcare_sql), "healthcare services")
    else:
        print("Import skipped:", row_count, "rows already present")

Imported: 4 healthcare services


## Cloud SQL analysis with BigQuery

The funding dataset was also imported into Google BigQuery.

- Project: `cidff-data-analysis`
- Dataset: `cidff_analysis`
- Table: `state_funding`
- Dataset location: EU
- Coverage: 2020–2024, across four departments
- Number of records: 30,309

Annual record counts and payment totals were checked against MySQL.
A second query compared program 137 payments with payments to CIDFF Rhône–Arc Alpin.

This step demonstrates SQL analysis in a cloud data warehouse.
The dataset itself is small and is not described as a large-scale Big Data dataset.

### Python extraction from BigQuery

The script `extract_bigquery.py` queries BigQuery from Google Cloud Shell.
It uses Google Cloud authentication without a password in the code.

The results are saved as `bigquery_annual_funding.csv`.
Annual record counts and payment totals match MySQL for all five years.

To run the script again in an authenticated environment:
`python3 extract_bigquery.py`

In [28]:
from pathlib import Path

# Add rules without deleting any existing content
with Path(".gitignore").open("a", encoding="utf-8") as file:
    file.write("\n# Local credentials\n.env\n")
    file.write("\n# Python cache\n__pycache__/\n*.pyc\n")

print("Updated: .gitignore")

Updated: .gitignore


In [29]:
from getpass import getpass
from secrets import token_urlsafe
from dotenv import set_key
from pathlib import Path

# Create local settings only if the file does not exist
env_path = Path(".env")

if env_path.exists():
    print("Skipped: .env already exists")
else:
    password = getpass("MySQL password: ")

    env_path.touch()
    set_key(env_path, "DB_USER", "root")
    set_key(env_path, "DB_PASSWORD", password)
    set_key(env_path, "DB_HOST", "localhost")
    set_key(env_path, "DB_NAME", "cidff_project")
    set_key(env_path, "API_KEY", token_urlsafe(32))

    print("Created: .env")

MySQL password:  ········


Created: .env


In [30]:
import requests
from dotenv import dotenv_values

# Read the key without displaying it
settings = dotenv_values(".env")

# Call the protected route with the correct key
response = requests.get(
    "http://127.0.0.1:8000/protected-check",
    headers={"X-API-Key": settings["API_KEY"]},
    timeout=10
)

print("Status:", response.status_code)
print(response.json())

Status: 200
{'message': 'Access granted'}


In [31]:
# Request the funding summary for 2024
response = requests.get(
    "http://127.0.0.1:8000/funding-summary",
    params={"year": 2024},
    headers={"X-API-Key": settings["API_KEY"]},
    timeout=10
)

print("Status:", response.status_code)

if response.status_code == 200:
    display(pd.DataFrame(response.json()))
else:
    print(response.text)

Status: 200


,department_code,record_count,total_amount
0,38,1722,1.095228e+08
1,69,3185,3.748937e+08
2,73,651,4.196415e+07
3,74,804,7.410467e+07


In [32]:
# Check that an incorrect key cannot access funding data
response = requests.get(
    "http://127.0.0.1:8000/funding-summary",
    params={"year": 2024},
    headers={"X-API-Key": "incorrect-key"},
    timeout=10
)

print("Status:", response.status_code)
print(response.json())

Status: 401
{'detail': 'Missing or invalid API key'}


In [33]:
# Check that a year outside the dataset coverage is rejected
response = requests.get(
    "http://127.0.0.1:8000/funding-summary",
    params={"year": 2025},
    headers={"X-API-Key": settings["API_KEY"]},
    timeout=10
)

print("Status:", response.status_code)

Status: 422


## Local API

The FastAPI application in `api.py` provides read access to MySQL.

- `/`: checks that the API is running.
- `/protected-check`: checks API key authentication.
- `/funding-summary`: returns payment counts and amounts by department
  for a selected year between 2020 and 2024.
- `/docs`: provides interactive OpenAPI documentation.

Credentials are stored in `.env`, which is excluded from Git.
Protected routes require an `X-API-Key` header.
The SQL query uses a bound parameter for the year.

### Validation

- Correct key: HTTP 200 and data returned from MySQL.
- Missing or incorrect key: HTTP 401.
- Year outside 2020–2024: HTTP 422.

The API runs locally and is not deployed on the Internet.
Amounts cover recorded state payments, not all association resources.

In [34]:
# Create a configuration example without real credentials
example_settings = """DB_USER=root
DB_PASSWORD=replace_with_your_mysql_password
DB_HOST=localhost
DB_NAME=cidff_project
API_KEY=replace_with_a_random_access_key
"""

Path(".env.example").write_text(example_settings, encoding="utf-8")

print("Created: .env.example")

Created: .env.example


In [35]:
import json

# Save the API specification without credentials
response = requests.get(
    "http://127.0.0.1:8000/openapi.json",
    timeout=10
)
response.raise_for_status()

Path("openapi.json").write_text(
    json.dumps(response.json(), indent=2),
    encoding="utf-8"
)

print("Saved: openapi.json")

Saved: openapi.json


## Conceptual Data Model — Merise

The conceptual model describes the main entities and their relationships.

It separates:
- associations identified in the RNA;
- organizations listed in the additional directory;
- healthcare services;
- recorded state payments;
- departmental violence indicators.

Directory coverage describes the departments served by an organization.
It does not necessarily describe its head office location.

Funding records are not automatically linked to RNA associations.
An exact SIRET match is available for only part of the dataset.

### Relationships and cardinalities

These rules describe the data stored in this project.

| First entity | Cardinality | Relationship | Cardinality | Second entity |
|---|---|---|---|---|
| Department | 0,N | contains the registered location of | 1,1 | RNA association |
| Department | 0,N | contains | 1,1 | Healthcare service |
| Department | 0,N | locates the beneficiary of | 1,1 | State payment |
| Department | 0,N | has | 1,1 | Annual violence observation |
| Department | 0,N | has | 1,1 | Minor violence observation |
| Department | 0,N | is covered by | 1,N | Directory organization |

Each directory organization covers at least one department.
Several organizations can cover the same department.

A payment is located using the beneficiary's recorded address.
This does not identify all territories where the funded action takes place.

### Entity definitions

| Entity | One record represents | Identifier |
|---|---|---|
| Department | One department | Department code |
| RNA association | One association in our RNA selection | RNA ID |
| Directory organization | One entry in the additional directory | Project directory ID |
| Healthcare service | One service listed in the source directory | Source service ID |
| State payment | One payment record in the funding files | Project payment ID |
| Annual violence observation | One department, year and violence type | Department + year + violence type |
| Minor violence observation | One department, period and violence type | Department + period + violence type |

Project IDs identify records within this database.
They are not official organization identifiers.

Minor violence observations contain average annual rates for 2022–2024.
They must not be treated as separate annual observations.

### Conceptual model diagram

![Conceptual data model](cidff_conceptual_model.png)

### From the conceptual model to MySQL

The conceptual model describes entities, relationships and cardinalities.
The physical model shows the tables, columns, data types and keys in MySQL.

For one-to-many relationships, the department code is stored as a foreign key.

Directory organizations can cover several departments, and each department
can be covered by several organizations. This many-to-many relationship
uses the linking table `directory_departments`.

Funding records remain separate from RNA associations because identifier
coverage is incomplete. Exact SIRET matches are used only when available.

In [37]:
# Read column definitions from the MySQL database
query = """
SELECT
    TABLE_NAME AS table_name,
    COLUMN_NAME AS column_name,
    COLUMN_TYPE AS data_type,
    IS_NULLABLE AS allows_null,
    COLUMN_KEY AS key_type
FROM information_schema.COLUMNS
WHERE TABLE_SCHEMA = 'cidff_project'
ORDER BY TABLE_NAME, ORDINAL_POSITION
"""

data_dictionary = pd.read_sql(query, engine)

print("Tables:", data_dictionary["table_name"].nunique())
print("Columns:", len(data_dictionary))
display(data_dictionary.head(10))

Tables: 8
Columns: 61


,table_name,column_name,data_type,allows_null,key_type
0,associations,rna_id,varchar(20),NO,PRI
1,associations,siret,char(14),YES,
2,associations,association_name,text,NO,
3,associations,association_purpose,text,YES,
4,associations,creation_date,date,YES,
5,associations,creation_year,smallint,YES,
6,associations,municipality_code,char(5),YES,
7,associations,municipality_name,varchar(255),YES,
8,associations,postal_code,char(5),YES,
9,associations,department_code,char(2),NO,MUL


In [38]:
# List the columns in each table
for table_name, columns in data_dictionary.groupby("table_name"):
    print("\n" + table_name)
    print(columns["column_name"].tolist())


associations
['rna_id', 'siret', 'association_name', 'association_purpose', 'creation_date', 'creation_year', 'municipality_code', 'municipality_name', 'postal_code', 'department_code', 'website', 'administrative_status', 'vss_vif', 'women_equality']

department_violence
['department_code', 'region_code', 'year', 'violence_type', 'count_unit', 'recorded_victims', 'rate_per_1000', 'population', 'population_reference_year', 'housing_units', 'housing_reference_year']

departments
['department_code', 'department_name']

directory_departments
['directory_id', 'department_code']

directory_organizations
['directory_id', 'organization_name', 'data_source', 'rna_id', 'head_office_verified']

funding_payments
['funding_id', 'year', 'department_code', 'municipality_code', 'municipality_name', 'siren', 'nic', 'association_name', 'budget_program', 'amount', 'funding_purpose', 'agreement', 'siren_original', 'siren_corrected', 'siren_format_valid', 'geography_corrected', 'identical_record_flag', 's

In [39]:
# Describe the fields used in the database
descriptions = {
    "rna_id": "Official RNA association identifier.",
    "siret": "14-digit identifier of an establishment.",
    "siren": "9-digit identifier of a legal entity.",
    "nic": "5-digit establishment code used with the SIREN.",
    "association_name": "Association or beneficiary name recorded in the source.",
    "association_purpose": "Declared association purpose recorded in the RNA.",
    "creation_date": "Association creation date recorded in the RNA.",
    "creation_year": "Year extracted from the association creation date.",
    "municipality_code": "INSEE municipality code of the recorded address.",
    "municipality_name": "Municipality name of the recorded address.",
    "postal_code": "Postal code of the recorded address.",
    "department_code": "Department code used to locate the record.",
    "department_name": "Department name.",
    "website": "Website address recorded in the source.",
    "administrative_status": "Association status code recorded in the RNA.",
    "vss_vif": "Keyword-based flag for sexual or domestic violence topics.",
    "women_equality": "Keyword-based flag for women or gender equality topics.",
    "region_code": "Region code recorded in the source.",
    "year": "Year covered by the observation or payment record.",
    "violence_type": "Category of violence measured in the source.",
    "count_unit": "Unit used to count recorded offences or victims.",
    "recorded_victims": "Number of victims recorded by police and gendarmerie.",
    "rate_per_1000": "Recorded rate per 1,000 inhabitants.",
    "population": "Population denominator provided in the source.",
    "population_reference_year": "Reference year of the population denominator.",
    "housing_units": "Number of housing units provided in the source.",
    "housing_reference_year": "Reference year of the housing denominator.",
    "directory_id": "Project identifier of an additional directory entry.",
    "organization_name": "Organization name in the additional directory.",
    "data_source": "Source used to identify the directory organization.",
    "head_office_verified": "Flag indicating whether head office verification was recorded.",
    "funding_id": "Project identifier of a funding record.",
    "budget_program": "State budget program code.",
    "amount": "Recorded state payment amount in euros.",
    "funding_purpose": "Payment purpose recorded for the year concerned.",
    "agreement": "Agreement information recorded in the source.",
    "siren_original": "SIREN value retained before correction.",
    "siren_corrected": "Flag indicating that the SIREN was corrected.",
    "siren_format_valid": "Flag indicating a valid 9-digit SIREN format, not verified identity.",
    "geography_corrected": "Flag indicating that shifted geographic fields were corrected.",
    "identical_record_flag": "Flag for identical records retained because duplicate payments are unconfirmed.",
    "historical_purpose_context": "Historical purpose used as context, not as the confirmed current purpose.",
    "directory_service_id": "Healthcare service identifier from the source directory.",
    "service_name": "Healthcare service name.",
    "period": "Period covered by the observation.",
    "rate_per_1000_minors": "Average annual recorded victim rate per 1,000 minors over the period."
}

# Match each column with its definition
data_dictionary["description"] = (
    data_dictionary["column_name"].map(descriptions)
)

print("Fields without a description:",
      data_dictionary["description"].isna().sum())

Fields without a description: 0


In [40]:
# Save the database field definitions
data_dictionary.to_csv(
    "data/processed/data_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", len(data_dictionary), "field definitions")

Saved: 61 field definitions
